# 🩺 Skin Cancer Detection – Complete ML/DL Pipeline (Google Colab)

**Dataset:** `fatemehmehrparvar/skin-cancer-detection` (Kaggle, downloaded with `kagglehub`)

**Before you run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

### Pipeline (simple view)
| Step | What happens | Easy example |
|---|---|---|
| 1. Dataset Acquisition | Download the images | Like downloading a folder of photos |
| 2. Dataset Visualization | Look at the data | See how many benign vs malignant photos |
| 3. Dataset Cleaning | Remove bad data | Delete broken or duplicate photos |
| 4. Feature Engineering | Turn images into model input | Resize every photo to 224×224, extract color numbers |
| 5. Dataset Splitting | Train / Validation / Test | Study material, practice quiz, final exam |
| 6. Model Training | Teach the models | Students learning from study material |
| 7. Model Testing | Check on unseen data + confusion matrix | The final exam |
| 8. Results Comparison | Compare all models | A scoreboard |
| 9. Extra: Prediction samples | See right/wrong predictions | Check the exam papers |
| 10. Download for Streamlit | Save best model + app files | Pack everything for the website |

> ⚠️ This project is for **learning only**. It is not a medical tool and must not be used for real diagnosis.

## ⚙️ Setup: install, imports and settings
**What this does:** loads all libraries and sets the main settings in one place.
**Example:** if you want faster training, change `EPOCHS = 15` to `EPOCHS = 5`.

In [ ]:
!pip install -q kagglehub

import os, json, shutil, hashlib, random, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from joblib import Parallel, delayed

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# ---------- Main settings (change here) ----------
SEED        = 42
IMG_SIZE    = 224      # all images are resized to 224 x 224
BATCH_SIZE  = 32
EPOCHS      = 15       # max epochs (early stopping may stop sooner)
FT_EPOCHS   = 5        # extra fine-tuning epochs for transfer-learning models
MAX_PER_CLASS = None   # e.g. 1500 to use fewer images per class for a quick test
# --------------------------------------------------

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU found :", tf.config.list_physical_devices("GPU"))

## 1️⃣ Dataset Acquisition
**What this does:** downloads the dataset from Kaggle and builds a table (DataFrame) with one row per image: its file path and its label.
**Example:** an image inside the folder `malignant/` gets the label `malignant`.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fatemehmehrparvar/skin-cancer-detection")
print("Path to dataset files:", path)

In [ ]:
# Look at the folder structure (first 3 levels)
for root, dirs, files in os.walk(path):
    depth = root[len(path):].count(os.sep)
    if depth <= 3:
        print(f"{'   ' * depth}{os.path.basename(root) or root}/   ({len(files)} files)")

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

records = []
for root, _, files in os.walk(path):
    for f in files:
        if os.path.splitext(f)[1].lower() in IMG_EXT:
            records.append({"path": os.path.join(root, f),
                            "label": os.path.basename(root).strip().lower()})  # label = folder name

df = pd.DataFrame(records)
print("Total images found:", len(df))
print("Classes found     :", df["label"].unique().tolist())

if df["label"].nunique() < 2:
    print("\n⚠️ Only one class found. Check the folder tree above: the labels may come from a CSV file "
          "instead of folder names. Adjust the label-building code in this cell if needed.")

# Optional: use fewer images per class (useful for a quick test run)
if MAX_PER_CLASS:
    df = df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED)).reset_index(drop=True)
    print("Using a subset:", len(df), "images")

df.head()

## 2️⃣ Dataset Visualization
**What this does:** helps us *see* the data before touching it.
- **Class balance** – example: 1800 benign vs 1500 malignant means the classes are almost balanced.
- **Sample images** – we check that the photos look right.
- **Image sizes** – example: if sizes differ, we must resize them later.
- **Color distribution** – malignant and benign lesions often differ slightly in color.

In [ ]:
# --- Class distribution ---
counts = df["label"].value_counts()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette="Set2")
ax[0].set_title("Images per class")
for i, v in enumerate(counts.values):
    ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", colors=sns.color_palette("Set2"))
ax[1].set_title("Class share")
plt.tight_layout(); plt.show()
print(counts)

In [ ]:
# --- Sample images from every class ---
classes = sorted(df["label"].unique())
n = 6
fig, axes = plt.subplots(len(classes), n, figsize=(2.4 * n, 2.6 * len(classes)), squeeze=False)
for r, c in enumerate(classes):
    sample = df[df["label"] == c].sample(n, random_state=SEED, replace=len(df[df["label"] == c]) < n)
    for k, p in enumerate(sample["path"]):
        axes[r, k].imshow(Image.open(p).convert("RGB"))
        axes[r, k].axis("off")
        if k == 0: axes[r, k].set_title(c, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# --- Image size check (on a sample of up to 400 images) ---
s = df.sample(min(400, len(df)), random_state=SEED)
sizes = [Image.open(p).size for p in s["path"]]
sz = pd.DataFrame(sizes, columns=["width", "height"])
print(sz.describe().loc[["min", "mean", "max"]].round(1))
print("Unique sizes in sample:", sz.drop_duplicates().shape[0])

# --- Average color per class (R, G, B) ---
rows = []
for _, r in s.iterrows():
    arr = np.asarray(Image.open(r["path"]).convert("RGB").resize((64, 64)), dtype=np.float32)
    rows.append({"label": r["label"], "R": arr[..., 0].mean(), "G": arr[..., 1].mean(), "B": arr[..., 2].mean()})
col = pd.DataFrame(rows).groupby("label").mean()
col.plot(kind="bar", color=["#d9534f", "#5cb85c", "#428bca"], figsize=(7, 4), title="Average color per class")
plt.ylabel("Average pixel value (0-255)"); plt.xticks(rotation=0); plt.show()

## 3️⃣ Dataset Cleaning
**What this does:** removes data that would hurt the models.
- **Corrupted images** – files that cannot be opened. *Example: a half-downloaded JPG.*
- **Tiny images** – too small to contain useful detail.
- **Exact duplicates** – the same photo twice. *Example: if one copy is in train and one in test, the test score becomes fake-high (data leakage).*
- **Conflicting labels** – the same photo labelled as two different classes, so we remove it.

In [ ]:
def inspect_image(p):
    try:
        with Image.open(p) as im:
            im.verify()                      # checks that the file is not broken
        with Image.open(p) as im:
            w, h = im.size
        with open(p, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()   # fingerprint to find duplicates
        return w, h, md5, True
    except Exception:
        return None, None, None, False

info = Parallel(n_jobs=-1)(delayed(inspect_image)(p) for p in tqdm(df["path"], desc="Checking images"))
df[["width", "height", "md5", "valid"]] = pd.DataFrame(info, index=df.index)

n0 = len(df)
print("Corrupted images :", (~df["valid"]).sum())
df = df[df["valid"]].copy()

tiny = (df["width"] < 32) | (df["height"] < 32)
print("Tiny images      :", tiny.sum())
df = df[~tiny]

conflict = df.groupby("md5")["label"].transform("nunique") > 1
print("Conflicting labels:", conflict.sum())
df = df[~conflict]

dups = df.duplicated("md5").sum()
print("Exact duplicates :", dups)
df = df.drop_duplicates("md5")

df = df.reset_index(drop=True)
print(f"\nImages before: {n0}  ->  after cleaning: {len(df)}")
print(df["label"].value_counts())

In [ ]:
# Encode labels: text -> number (example: benign=0, malignant=1)
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])
class_names = list(le.classes_)
NUM_CLASSES = len(class_names)
print("Classes:", dict(zip(class_names, range(NUM_CLASSES))))

# Positive class = the dangerous one (used for sensitivity / recall of cancer)
POS_IDX = next((i for i, c in enumerate(class_names) if "malig" in c), 1 if NUM_CLASSES == 2 else 0)
print("Positive class used for sensitivity:", class_names[POS_IDX])

## 4️⃣ Feature Engineering
**What this does:** turns images into numbers the models can learn from. We use two ways:

**A) Deep learning (CNN):** images are resized to 224×224 and fed directly. The CNN learns its own features.
We also add **data augmentation** (random flip, rotation, zoom, contrast) so the model sees slightly different versions of each photo. *Example: a lesion flipped left-right is still the same lesion.*

**B) Classical ML (Random Forest, SVM):** these need a table of numbers, so we create **hand-made features** from each image:
color histograms, average/standard-deviation of colors (RGB and HSV), edge strength, and centre-vs-border color difference.
*Example: malignant lesions are often darker and more uneven in color, so color spread is a useful number.*

In [ ]:
# ---------- A) Data augmentation (used inside the CNN models, only during training) ----------
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

# ---------- B) Hand-made features for classical ML ----------
def extract_features(p, size=64):
    im = Image.open(p).convert("RGB").resize((size, size))
    rgb = np.asarray(im, dtype=np.float32)
    hsv = np.asarray(im.convert("HSV"), dtype=np.float32)
    gray = rgb.mean(axis=2)
    f = []
    for ch in range(3):                                   # color histograms (16 bins x 3 channels)
        hist, _ = np.histogram(rgb[..., ch], bins=16, range=(0, 255), density=True)
        f.extend(hist)
    f.extend(rgb.mean((0, 1))); f.extend(rgb.std((0, 1)))  # RGB mean / std
    f.extend(hsv.mean((0, 1))); f.extend(hsv.std((0, 1)))  # HSV mean / std
    gy, gx = np.gradient(gray)
    mag = np.hypot(gx, gy)
    f.extend([mag.mean(), mag.std()])                     # edge strength
    c = size // 4
    centre = rgb[c:-c, c:-c].reshape(-1, 3).mean(0)
    border = np.concatenate([rgb[:c].reshape(-1, 3), rgb[-c:].reshape(-1, 3)]).mean(0)
    f.extend(centre - border)                             # centre vs border color
    return np.array(f, dtype=np.float32)

feats = Parallel(n_jobs=-1)(delayed(extract_features)(p) for p in tqdm(df["path"], desc="Extracting features"))
X_feat = np.vstack(feats)
print("Hand-made feature table:", X_feat.shape, "(images x features)")

## 5️⃣ Dataset Splitting
**What this does:** splits the data into 3 parts (70% / 15% / 15%).
- **Train** – the model learns from it (*study material*).
- **Validation** – used while training to tune and stop early (*practice quiz*).
- **Test** – used only at the end (*final exam*, never seen before).

We use **stratified** splitting so every part keeps the same benign/malignant ratio.

In [ ]:
idx = np.arange(len(df))
idx_train, idx_tmp = train_test_split(idx, test_size=0.30, stratify=df["y"], random_state=SEED)
idx_val, idx_test  = train_test_split(idx_tmp, test_size=0.50, stratify=df["y"].values[idx_tmp], random_state=SEED)

paths = df["path"].values
y = df["y"].values

train_paths, y_train = paths[idx_train], y[idx_train]
val_paths,   y_val   = paths[idx_val],   y[idx_val]
test_paths,  y_test  = paths[idx_test],  y[idx_test]

X_feat_train, X_feat_val, X_feat_test = X_feat[idx_train], X_feat[idx_val], X_feat[idx_test]

split_df = pd.DataFrame({
    "Train": pd.Series(y_train).map(dict(enumerate(class_names))).value_counts(),
    "Validation": pd.Series(y_val).map(dict(enumerate(class_names))).value_counts(),
    "Test": pd.Series(y_test).map(dict(enumerate(class_names))).value_counts()})
print(split_df)
split_df.T.plot(kind="bar", stacked=True, figsize=(7, 4), title="Class balance in each split", colormap="Set2")
plt.xticks(rotation=0); plt.show()

# tf.data pipelines: read image -> resize -> batch (fast loading for the CNN models)
AUTOTUNE = tf.data.AUTOTUNE

def load_img(p, label):
    img = tf.io.read_file(p)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.clip_by_value(img, 0, 255), tf.uint8), label

def make_ds(p, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((p, labels)).map(load_img, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(1024, seed=SEED)
    ds = ds.batch(BATCH_SIZE).map(lambda x, l: (tf.cast(x, tf.float32), l), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_paths, y_train, training=True)
val_ds   = make_ds(val_paths, y_val)
test_ds  = make_ds(test_paths, y_test)      # no shuffle -> predictions stay in the same order as y_test

# Class weights: give the smaller class more importance (handles imbalance)
cw = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = {int(c): float(w) for c, w in zip(np.unique(y_train), cw)}
print("Class weights:", class_weights)

## 6️⃣ Model Training
We train **5 models** so we can compare them:

| Model | Type | Easy explanation |
|---|---|---|
| Custom CNN | Deep learning | A small network built from scratch |
| MobileNetV2 | Transfer learning | A network already trained on millions of photos (ImageNet), then adapted to skin images |
| EfficientNetB0 | Transfer learning | Same idea, a stronger pretrained network |
| Random Forest | Classical ML | Many decision trees voting on the hand-made features |
| SVM | Classical ML | Finds the best boundary between classes in the hand-made features |

**Transfer learning example:** a student who already knows how to recognise shapes and textures learns skin cancer faster than a student starting from zero.
Training uses **early stopping** (stop when validation stops improving) and **learning-rate reduction** (take smaller steps when stuck).

In [ ]:
all_results = {}   # name -> dict with metrics, predictions, probabilities
dl_models   = {}   # name -> trained keras model
histories   = {}   # name -> training history

def evaluate(name, y_true, y_pred, probs):
    avg = "binary" if NUM_CLASSES == 2 else "weighted"
    kw = dict(pos_label=POS_IDX) if NUM_CLASSES == 2 else {}
    if NUM_CLASSES == 2:
        auc = roc_auc_score(y_true, probs[:, POS_IDX])
    else:
        auc = roc_auc_score(y_true, probs, multi_class="ovr")
    m = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0, **kw),
        "Recall (Sensitivity)": recall_score(y_true, y_pred, average=avg, zero_division=0, **kw),
        "F1-score":  f1_score(y_true, y_pred, average=avg, zero_division=0, **kw),
        "ROC-AUC":   auc,
    }
    all_results[name] = {"metrics": m, "y_pred": y_pred, "probs": probs}
    return m

def make_callbacks():
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)]

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def train_dl(name, model, base=None, lr=1e-3):
    print(f"\n================ Training {name} ================")
    compile_model(model, lr)
    h = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                  class_weight=class_weights, callbacks=make_callbacks(), verbose=2)
    hist = {k: list(v) for k, v in h.history.items()}
    if base is not None and FT_EPOCHS > 0:          # fine-tuning: unfreeze the last layers of the pretrained net
        print(f"---- Fine-tuning {name} ----")
        base.trainable = True
        for layer in base.layers[:-30]:
            layer.trainable = False
        compile_model(model, 1e-5)
        h2 = model.fit(train_ds, validation_data=val_ds, epochs=FT_EPOCHS,
                       class_weight=class_weights, callbacks=make_callbacks(), verbose=2)
        for k in hist: hist[k] += list(h2.history[k])
    dl_models[name], histories[name] = model, hist
    return model

def build_custom_cnn():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inp)
    x = layers.Rescaling(1. / 255)(x)
    for f in [32, 64, 128, 256]:
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="Custom_CNN")

def build_transfer(name, base_cls, preprocess=None):
    base = base_cls(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augmentation(inp)
    if preprocess is not None:
        x = preprocess(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name=name), base

# Models take raw 0-255 pixels; scaling is done INSIDE the model (makes Streamlit deployment easy)

In [ ]:
# ----- Model 1: Custom CNN -----
train_dl("Custom CNN", build_custom_cnn())

In [ ]:
# ----- Model 2: MobileNetV2 (transfer learning) -----
m, base = build_transfer("MobileNetV2", keras.applications.MobileNetV2,
                         preprocess=layers.Rescaling(1. / 127.5, offset=-1))   # MobileNetV2 expects [-1, 1]
train_dl("MobileNetV2", m, base)

In [ ]:
# ----- Model 3: EfficientNetB0 (transfer learning) -----
m, base = build_transfer("EfficientNetB0", keras.applications.EfficientNetB0)  # has its own scaling inside
train_dl("EfficientNetB0", m, base)

In [ ]:
# ----- Model 4 & 5: Classical ML on the hand-made features -----
scaler = StandardScaler().fit(X_feat_train)         # fit on TRAIN only (no leakage)
Xtr, Xva, Xte = scaler.transform(X_feat_train), scaler.transform(X_feat_val), scaler.transform(X_feat_test)

rf = RandomForestClassifier(n_estimators=400, class_weight="balanced", n_jobs=-1, random_state=SEED).fit(Xtr, y_train)
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True, class_weight="balanced", random_state=SEED).fit(Xtr, y_train)
print("Random Forest  -> validation accuracy:", round(rf.score(Xva, y_val), 4))
print("SVM            -> validation accuracy:", round(svm.score(Xva, y_val), 4))

## 7️⃣ Model Testing (with Confusion Matrix)
**What this does:** every model takes the **final exam** on the test set it has never seen.

**How to read a confusion matrix (binary example):**
| | Predicted benign | Predicted malignant |
|---|---|---|
| **Actually benign** | ✅ True Negative | ❌ False Positive (false alarm) |
| **Actually malignant** | ❌ **False Negative (missed cancer – the most dangerous error)** | ✅ True Positive |

So for cancer detection, a high **Recall / Sensitivity** (few missed cancers) is especially important.

In [ ]:
# Predict on the test set
for name, model in dl_models.items():
    probs = model.predict(test_ds, verbose=0)
    evaluate(name, y_test, probs.argmax(1), probs)

for name, mdl in [("Random Forest", rf), ("SVM", svm)]:
    probs = mdl.predict_proba(Xte)
    evaluate(name, y_test, probs.argmax(1), probs)

print("Done. Models tested:", list(all_results))

In [ ]:
# Confusion matrix + classification report for every model
names = list(all_results)
cols = 3
rows = int(np.ceil(len(names) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 4.6 * rows))
axes = np.array(axes).reshape(-1)
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, all_results[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"{name}\nAccuracy: {all_results[name]['metrics']['Accuracy']:.3f}")
for ax in axes[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

for name in names:
    print(f"\n===== {name} =====")
    print(classification_report(y_test, all_results[name]["y_pred"], target_names=class_names, digits=3))

## 8️⃣ Results Visualization and Comparison
**What this does:** a scoreboard for all models, plus training curves and ROC curves.
- **Accuracy** – how many predictions were right overall.
- **Precision** – of the images predicted as cancer, how many really were. *(low = many false alarms)*
- **Recall (Sensitivity)** – of the real cancers, how many we found. *(low = missed cancers)*
- **F1-score** – balance of precision and recall.
- **ROC-AUC** – how well the model separates the classes (1.0 = perfect, 0.5 = random guessing).

In [ ]:
results_df = pd.DataFrame({n: r["metrics"] for n, r in all_results.items()}).T
results_df = results_df.sort_values("F1-score", ascending=False)
display(results_df.round(4).style.background_gradient(cmap="Greens", axis=0))

results_df.plot(kind="bar", figsize=(12, 5), colormap="Set2", width=0.8)
plt.title("Model comparison on the test set"); plt.ylim(0, 1.05); plt.xticks(rotation=20)
plt.legend(loc="lower right", ncol=3); plt.tight_layout(); plt.show()

In [ ]:
# Training curves (accuracy + loss) for the deep learning models
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories.items():
    ax[0].plot(h["val_accuracy"], label=name)
    ax[1].plot(h["val_loss"], label=name)
ax[0].set_title("Validation accuracy per epoch"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].set_title("Validation loss per epoch");     ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ROC curves (for 2-class problems)
if NUM_CLASSES == 2:
    plt.figure(figsize=(7, 6))
    for name, r in all_results.items():
        fpr, tpr, _ = roc_curve(y_test, r["probs"][:, POS_IDX], pos_label=POS_IDX)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {r['metrics']['ROC-AUC']:.3f})")
    plt.plot([0, 1], [0, 1], "k--", label="Random guess")
    plt.xlabel("False Positive Rate (false alarms)"); plt.ylabel("True Positive Rate (cancers found)")
    plt.title("ROC curves"); plt.legend(loc="lower right"); plt.show()

## 9️⃣ Extra Step: Prediction Samples and Errors
**What this does:** shows real test images with the true label and the best model's prediction (green = correct, red = wrong).
Looking at mistakes helps us understand the model. *Example: if many errors are very dark or blurry photos, better data would help.*

In [ ]:
dl_results = results_df.loc[[n for n in results_df.index if n in dl_models]]
best_name = dl_results["F1-score"].idxmax()
best_model = dl_models[best_name]
print("🏆 Best deep learning model (by test F1-score):", best_name)
print(dl_results.loc[best_name].round(4).to_dict())

pred = all_results[best_name]["y_pred"]
probs = all_results[best_name]["probs"]
pick = np.random.RandomState(SEED).choice(len(test_paths), size=min(12, len(test_paths)), replace=False)
fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(Image.open(test_paths[i]).convert("RGB")); ax.axis("off")
    ok = pred[i] == y_test[i]
    ax.set_title(f"True: {class_names[y_test[i]]}\nPred: {class_names[pred[i]]} ({probs[i].max():.0%})",
                 color="green" if ok else "red", fontsize=10)
plt.suptitle(f"{best_name} predictions on test images", fontsize=14); plt.tight_layout(); plt.show()

## 🔟 Best Model Download for Streamlit Cloud Deployment
**What this does:** saves the best deep learning model and creates everything you need for Streamlit Cloud:
| File | Purpose |
|---|---|
| `best_model.keras` | The trained model |
| `class_names.json` | The label names (example: `["benign", "malignant"]`) |
| `app.py` | The Streamlit web app (upload a photo, get a prediction) |
| `requirements.txt` | Libraries to install on Streamlit Cloud (versions match this Colab) |

Everything is zipped into **`streamlit_deployment.zip`** and downloaded automatically.

In [ ]:
DEPLOY_DIR = "streamlit_deployment"
shutil.rmtree(DEPLOY_DIR, ignore_errors=True)
os.makedirs(DEPLOY_DIR)

best_model.save(f"{DEPLOY_DIR}/best_model.keras")
with open(f"{DEPLOY_DIR}/class_names.json", "w") as f:
    json.dump(class_names, f)

with open(f"{DEPLOY_DIR}/requirements.txt", "w") as f:
    f.write(f"streamlit\ntensorflow-cpu=={tf.__version__}\nnumpy\npillow\n")

print("Saved model:", best_name, "->", os.path.getsize(f"{DEPLOY_DIR}/best_model.keras") // 1024, "KB")

In [ ]:
app_code = '''import json
import numpy as np
import streamlit as st
import tensorflow as tf
from PIL import Image

IMG_SIZE = 224

st.set_page_config(page_title="Skin Cancer Detection", page_icon="🩺")
st.title("🩺 Skin Cancer Detection")
st.caption("Educational demo only. This is NOT a medical diagnosis. Please see a doctor for any skin concern.")

@st.cache_resource
def load_model_and_labels():
    model = tf.keras.models.load_model("best_model.keras")
    with open("class_names.json") as f:
        class_names = json.load(f)
    return model, class_names

model, class_names = load_model_and_labels()

file = st.file_uploader("Upload a skin lesion image", type=["jpg", "jpeg", "png"])
if file is not None:
    img = Image.open(file).convert("RGB")
    st.image(img, caption="Uploaded image", use_container_width=True)

    # The model scales pixels internally, so we give it raw 0-255 values
    x = np.array(img.resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)[None, ...]
    probs = model.predict(x, verbose=0)[0]
    top = int(np.argmax(probs))

    st.subheader(f"Prediction: {class_names[top]}")
    st.write(f"Confidence: {probs[top]:.1%}")
    st.bar_chart({c: float(p) for c, p in zip(class_names, probs)})
'''
with open(f"{DEPLOY_DIR}/app.py", "w") as f:
    f.write(app_code)

# Zip everything and download
with zipfile.ZipFile("streamlit_deployment.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(DEPLOY_DIR):
        z.write(os.path.join(DEPLOY_DIR, fn), arcname=fn)
print("Files inside the zip:", os.listdir(DEPLOY_DIR))

from google.colab import files
files.download("streamlit_deployment.zip")

## 🚀 How to deploy on Streamlit Cloud (simple steps)
1. Unzip `streamlit_deployment.zip` and upload the 4 files to a **new GitHub repository**.
   *(If `best_model.keras` is larger than 100 MB, use Git LFS.)*
2. Go to **share.streamlit.io** → **New app** → pick your repository → main file: `app.py`.
3. In **Advanced settings**, choose **Python 3.11** (it works best with TensorFlow) and click **Deploy**.

**Quick local test (optional):** `pip install -r requirements.txt` then `streamlit run app.py`.